In [108]:
import httpx

In [109]:
scrape = httpx.post('http://localhost:3000/v1/scrape', json={"url": "https://en.wikipedia.org/wiki/QBism", "formats": ["plainText"]}, headers={"Content-Type": "application/json"})

In [95]:
search = httpx.post('http://localhost:3000/v1/search', json={"query": "asd", "limit": 5})

In [110]:
scrape.json()['data']
# search.json()['data']

{'plainText': 'Jump to content\nEach point in the Bloch ball is a possible quantum state for a qubit. In QBism, all quantum states are representations of personal probabilities.\nIn physics and the philosophy of physics, QBism (pronounced "cubism") is an interpretation of quantum mechanics that takes an agent\'s actions and experiences as the central concerns of the theory. It is the most prominent and extreme form of quantum Bayesianism, a collection of related approaches that all involve interpreting quantum probabilities as Bayesian in some manner. QBism deals with common questions in the interpretation of quantum theory about the nature of wavefunction superposition, quantum measurement, and entanglement.[1][2][3] According to QBism, many, but not all, aspects of the quantum formalism are subjective in nature. For example, in this interpretation, a quantum state is not an element of reality—instead, it represents the degrees of belief an agent has about the possible outcomes of mea

In [100]:
# import httpx
# from pydantic_ai import Agent, ModelRetry
# from pydantic_ai.models.openai import OpenAIChatModel
# from pydantic_ai.providers.openai import OpenAIProvider

# CRW_BASE = "http://localhost:3000"

# model = OpenAIChatModel(
#     "./Qwen3.6-35B-A3B-UD-IQ4_NL.gguf",
#     provider=OpenAIProvider(
#         base_url="http://localhost:8090/v1",
#         api_key="local",
#     ),
# )

# agent = Agent(
#     model,
#     instructions=(
#         "You are a web research assistant. You do NOT answer factual questions from "
#         "memory. Your workflow is always: (1) call `search` to find relevant pages, "
#         "(2) call `scrape` on the 1-3 most promising URLs to read their full text, "
#         "(3) answer using only what you scraped, and cite the source URLs. "
#         "If the first search is unhelpful, refine the query and search again."
#     ),
# )


# @agent.tool_plain
# def search(query: str, limit: int = 5) -> list[dict]:
#     """Search the web and return candidate results.

#     Returns a list of {title, url, snippet}. Use this to discover URLs worth scraping.
#     """
#     query = query.strip()
#     if not query:
#         raise ModelRetry("The search query was empty. Provide a specific query string.")
#     r = httpx.post(
#         f"{CRW_BASE}/v1/search",
#         json={"query": query, "limit": limit},
#         timeout=30)
#     r.raise_for_status()
#     results = r.json()["data"]["results"]
#     if not results:
#         raise ModelRetry(f"No results for {query!r}. Try broader or different keywords.")
#     return [{"title": x["title"], "url": x["url"], "snippet": x.get("snippet", "")} for x in results]


# @agent.tool_plain
# def scrape(url: str) -> str:
#     """Fetch a single web page and return its plain-text content. Pass a URL from `search`."""
#     if not url.startswith(("http://", "https://")):
#         raise ModelRetry(f"{url!r} is not a valid URL. Use a full http(s) URL from search results.")
#     r = httpx.post(
#         f"{CRW_BASE}/v1/scrape",
#         json={"url": url, "formats": ["plainText"]},
#         headers={"Content-Type": "application/json"},
#         timeout=60,
#     )
#     r.raise_for_status()
#     data = r.json()
#     # adjust the key if your fastcrw returns a different shape
#     text = data.get("data", {}).get("plainText") or ""
#     if not text.strip():
#         raise ModelRetry(f"Scraping {url} returned no text. Try a different URL.")
#     return text[:20000]  # trim so one page can't blow the context window

In [101]:
# from pydantic_ai.usage import UsageLimits

# response = await agent.run(
#     "What is QBism?",
#     usage_limits=UsageLimits(request_limit=6),  # cap model round-trips per run
# )

In [19]:
# print(response.output)

In [18]:
 # for m in response.all_messages():
 #      for part in m.parts:
 #          kind = part.part_kind
 #          if kind == "tool-call":
 #              print("→ CALL", part.tool_name, part.args)
 #          elif kind == "tool-return":
 #              print("← RET ", part.tool_name, str(part.content)[:120])

In [17]:
# response = await agent.run(
#     "What is the future for intel stocks look like ? Is it a good time to invest, because they are working on new technology, and new releases are coming close.",
#     usage_limits=UsageLimits(request_limit=6),  # cap model round-trips per run
# )

In [135]:
import datetime as dt
import time

from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel

import httpx

from pydantic import BaseModel, ValidationError

from src.config import settings
from src.errors import AppError, BadRequestError

CRW_BASE = "http://localhost:3000"

class AppError(Exception):
    """Unexpected internal server error"""
    status_code = 500
    detail = "Unexpected internal server error"

    def __init__(self, code: str, detail: str | None = None):
        self.code = code
        if detail is not None:
            self.detail = detail
        super().__init__(f"{code}: {self.detail}")


class SearchHit(BaseModel):
    model_config = {"extra": "ignore"}
    url: str
    title: str
    description: str
    snippet: str
    position: int
    score: float
    category: str

class SearchResponseData(BaseModel):
    results: list[SearchHit]

class SearchResponse(BaseModel):
    success: bool
    data: SearchResponseData


class CrwClient:
    def __init__(self):
        self._http = httpx.AsyncClient(base_url=CRW_BASE, timeout=30)

    async def search(self, query: str, limit: int = 5) -> list[SearchHit]:
        if not query:
            raise BadRequestError("CRWC-01", "Missing search query")

        try:
            r = await self._http.post("/v1/search", json={"query": query, "limit": limit})
            r.raise_for_status()
        except httpx.HTTPStatusError as exc:
            # request error or status 500
            raise AppError("CRWC-02", f"Crawler returned status {exc.response.status_code}") from exc
        except httpx.RequestError as exc:
            # unreachable
            raise AppError("CRWC-03", "Crawler backend unreachable") from exc

        try:
            search_response = SearchResponse(**r.json())
        except (ValidationError, ValueError) as exc:
            raise AppError("CRWC-04", "Crawler returned malformed data") from exc

        return search_response.data.results

WEB_QUERY_GENERATOR_AGENT_INSTRUCTIONS = (
    "You are a web search engine query generator."
    "Based on what you know and the given prompt, write 4 web search engine queries."
    "Let 1 query be for getting up to date information regarding the prompt."
    "Let the other 2 queries be for getting more general information that you don't know, if there is such."
)

web_query_generator_agent = Agent(
    instructions=WEB_QUERY_GENERATOR_AGENT_INSTRUCTIONS,
    output_type=list[str],
)


@web_query_generator_agent.tool_plain
def get_current_date() -> str:
    """Return today's date as an ISO string (YYYY-MM-DD), for building up-to-date queries."""
    return dt.date.today().isoformat()


CONTENT_RELEVANCY_AGENT_INSTRUCTIONS = (
    "You are content relevancy analyzer agent."
    "You will be given content to review, and judge if it is relevant for the prompt or no."
    "If content is relevant respond with True, and if not, respond with False"
)

content_relevancy_agent = Agent(
    instructions=CONTENT_RELEVANCY_AGENT_INSTRUCTIONS,
    output_type=str
)


async def enhance_prompt_with_search(prompt: str, model: OpenAIChatModel, crw_client: CrwClient):
    generated_queries_answer = await web_query_generator_agent.run(prompt, model=model)
    queries: list[str] = generated_queries_answer.output

    print(queries)

    search_results: list[SearchHit] = []
    for query in queries:
        result: list[SearchHit] = await crw_client.search(query, 2)
        search_results.extend(result)
        time.sleep(1)

    if len(search_results) == 0:
        raise AppError("WSP-01", "CRW Search results are empty, or CRW is unreachable")


    relevant_results: list[SearchHit] = []
    for search_result in search_results:
        content_for_review = (
            f"URL: {search_result.url}\n"
            f"Title: {search_result.title}\n"
            f"Description: {search_result.description}\n"
            f"Snippet: {search_result.snippet}"
        )
        message = (
            f"Original prompt:\n{prompt}\n\n"
            f"Content to review:\n{content_for_review}"
        )
        answer = await content_relevancy_agent.run(user_prompt=message, model=model)
        # print(f"generated prompt for llm: {message}, answer: {answer.output}")
        if answer.output:
            relevant_results.append(search_result)

    return relevant_results

In [136]:
result = await enhance_prompt_with_search("Who are shopi people?", model, CrwClient())

['Who are the Shopi people today?', 'Who are the Shopi people history?', 'Shopi people Albania origin', 'Shopi people geography']


In [137]:
# print(result)

In [133]:
import asyncio
import logging
import httpx

from pydantic import BaseModel, ValidationError

from src.config import settings
from src.errors import AppError, BadRequestError


logger = logging.getLogger(__name__)


CRW_BASE = "http://localhost:3000"


class SearchHit(BaseModel):
    model_config = {"extra": "ignore"}
    url: str
    title: str
    description: str
    snippet: str
    position: int
    score: float
    category: str

class SearchResponseData(BaseModel):
    results: list[SearchHit]

class SearchResponse(BaseModel):
    success: bool
    data: SearchResponseData

class RenderDecision(BaseModel):
    kind: str
    chosen: str

class ScrapedData(BaseModel):
    model_config = {"extra": "ignore"}
    plainText: str
    renderDecision: RenderDecision
    creditCost: int
    metadata: dict
    contentType: str
    url: str


class CrwClient:
    def __init__(self):
        self._http = httpx.AsyncClient(base_url=CRW_BASE, timeout=30)

    async def search(self, query: str, limit: int = 5) -> list[SearchHit]:
        if not query:
            raise BadRequestError("CRWC-01", "Missing search query")

        try:
            r = await self._http.post("/v1/search", json={"query": query, "limit": limit})
            r.raise_for_status()
        except httpx.HTTPStatusError as exc:
            # request error or status 500
            raise AppError("CRWC-02", f"Crawler search returned status {exc.response.status_code}") from exc
        except httpx.RequestError as exc:
            # unreachable
            raise AppError("CRWC-03", "Crawler search backend unreachable") from exc

        try:
            search_response = SearchResponse(**r.json())
        except (ValidationError, ValueError) as exc:
            raise AppError("CRWC-04", "Crawler search returned malformed data") from exc

        return search_response.data.results


    async def scrape(self, urls: list[str], formats: list[str] | None = None) -> list[ScrapedData] | None:
        if len(urls) == 0:
            raise BadRequestError("CRWC-09", "Missing request urls")
        if formats is None:
            formats = ['plainText']

        results = await asyncio.gather(*(
            self._http.post("/v1/scrape",
                            json={"url": url, "formats": formats},
                            headers={"Content-Type": "application/json"})
            for url in urls), return_exceptions=True)

        scraped_data_list: list[ScrapedData] = []
        for url, item in zip(urls, results):
            if isinstance(item, Exception):
                print(f"CRWC-10 Partial success: failed scrape for {url}: {item}")
                continue
            else:
                try:
                    parsed = ScrapedData(**{**item.json()["data"], "url": url})
                except (ValidationError, ValueError, TypeError) as exc:
                    print(f"CRWC-11 Partial success: CRW scraper returned malformed data for success {url}: {exc}")
                    continue

                scraped_data_list.append(parsed)

        return scraped_data_list if len(scraped_data_list) != 0 else None


crw_client = CrwClient()

In [138]:
#result = await crw_client.scrape(["https://en.wikipedia.org/wiki/QBism", "https://bigthink.com/13-8/qbism-quantum-reality/", "https://medium.com/quantum-mysteries/debating-quantum-reality-my-exchange-with-dr-christopher-fuchs-founder-of-qbism-3986867700fc"])
# print(result)




In [149]:
import asyncio
import zipfile
from typing import AsyncIterator

from fastapi import UploadFile
from pydantic import BaseModel
from pydantic_ai import Agent, BinaryContent
from pydantic_ai.models import Model


model = OpenAIChatModel(
    "./Qwen3.6-35B-A3B-UD-IQ4_NL.gguf",
    provider=OpenAIProvider(
        base_url="http://localhost:8090/v1",
        api_key="local",
    ),
)

_ZIP_SUFIX = (".zip",)

_JPG_SUFFIXES = (".jpg", ".jpeg")


class Transcription(BaseModel):
    filename: str
    text: str


IMAGE_TEXT_TO_TEXT_AGENT_INSTRUCTIONS = (
    "You are an image-to-text transcription agent."
    "Given an image containing text (handwritten or printed), output exactly what is written as a string."
    "Output only the transcribed text, nothing else."
    "If the image contains no text, output an empty string."
)

image_text_to_text_agent = Agent(
    instructions=IMAGE_TEXT_TO_TEXT_AGENT_INSTRUCTIONS,
    output_type=str,
)

async def convert_to_text(image_bytes: bytes, model: Model, media_type: str = "image/jpeg") -> str:
    result = await image_text_to_text_agent.run(
        ["Transcribe the text in this image.",
         BinaryContent(data=image_bytes, media_type=media_type)],
        model=model,
    )
    return result.output


async def transcribe_uploads(files: list[UploadFile], model: Model) -> list[Transcription]:
    """Expand any zips, transcribe each image one at a time, return results in order."""
    results: list[Transcription] = []
    async for name, image_bytes in _iter_images(files):
        text = await convert_to_text(image_bytes, model)
        if not text or len(text) == 0:
            continue
        results.append(Transcription(filename=name, text=text))
    return results


async def _iter_images(upload_files: list[UploadFile]) -> AsyncIterator[tuple[str, bytes]]:
    """Yield (name, jpg_bytes) one at a time from uploaded files or zips.

    A .zip is read from its spooled temp file (on disk for > 1MB uploads),
    decompressing one entry at a time; a .jpg is passed through.
    macOS zip junk (__MACOSX/, ._*) is skipped.
    """
    for f in upload_files:
        name = f.filename or "upload"
        lower = name.lower()
        if lower.endswith(_ZIP_SUFIX):
            await f.seek(0)
            zf = await asyncio.to_thread(zipfile.ZipFile, f.file)
            try:
                entries = [
                    n for n in zf.namelist()
                    if n.lower().endswith(_JPG_SUFFIXES)
                    and not n.startswith("__MACOSX/")
                    and not n.rsplit("/", 1)[-1].startswith("._")
                ]
                for entry in entries:
                    data = await asyncio.to_thread(zf.read, entry)
                    yield entry, data
            finally:
                zf.close()
        elif lower.endswith(_JPG_SUFFIXES):
            yield name, await f.read()  # already runs in a threadpool


In [150]:
import io
import sys
from pathlib import Path
from tempfile import SpooledTemporaryFile

from starlette.datastructures import UploadFile
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

from src.config import settings
from src.agents.text_image_to_image_services import transcribe_uploads

def load_uploads(folder: Path) -> list[UploadFile]:
      uploads: list[UploadFile] = []
      for path in sorted(folder.iterdir()):
          if path.is_file():
              uploads.append(
                  UploadFile(file=io.BytesIO(path.read_bytes()), filename=path.name)
              )
      return uploads

r = await transcribe_uploads(load_uploads(Path("test-images")), model=model)

In [153]:
for a in r:
    print(a.filename)
    print(a.text)
    print("\n")

BUihWvTM.jpg
Will it exist?
Will it ever happen, I don't
know
I might as well always
be sitting, riding, walking
looking at the window
wondering about time
where I am in it
what I am in it
and for how long.


HPwzIzDo.jpg
See it for what it is little gorgeous things...!
What has happened
Who am I in this point in time
Am I here, do I still notice
things can I still see them,
the others, can I still help
them
Am I too distant, distracted
My eyes begin to set on
what I imagine
A green country
whirly waves dancing
will I be able to kneel
and look at that creature
the one that would come
out of me, what wil it
know, what will it feel,
will it see those who are
gone? will it know where
I am coming from ?


y0cEeGlF.jpg
30/07/26
A funny little place I'm in
I feel a strange shift
happening
Like when they say
Imagine yourself next year
Six months in, I'm already
different
I didn't get to write all of it
Almost none of it
I hope the shift will
hurt me
I hope it won't let me go
I hope it slows d